# Professional Practice X4: Semi-Supervised Learning

Learn with limited labeled data using semi-supervised techniques.

**Goal:** Use unlabeled data to improve supervised learning.

**Techniques:**
- Self-training (pseudo-labeling)
- Label Propagation
- Combining clustering with supervised learning

**Use Case:** When labels are expensive but unlabeled data is abundant

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Load data
digits = load_digits()
X, y = digits.data, digits.target

# Split: train (labeled) + unlabeled + test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.7, random_state=42, stratify=y
)
X_unlabeled, X_test, y_unlabeled, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Simulate limited labels: use only 10% of training data
n_labeled = int(0.1 * len(X_train))
X_labeled = X_train[:n_labeled]
y_labeled = y_train[:n_labeled]

print(f"Labeled samples: {len(X_labeled)}")
print(f"Unlabeled samples: {len(X_unlabeled)}")
print(f"Test samples: {len(X_test)}")

# Baseline: Supervised learning with only labeled data
clf_supervised = LogisticRegression(max_iter=1000, random_state=42)
clf_supervised.fit(X_labeled, y_labeled)
acc_supervised = clf_supervised.score(X_test, y_test)
print(f"\nBaseline (supervised only): {acc_supervised:.4f}")

# Semi-supervised: Label Propagation
print("\n1. Label Propagation...")

# Combine labeled and unlabeled, mark unlabeled as -1
X_combined = np.vstack([X_labeled, X_unlabeled])
y_combined = np.hstack([y_labeled, -np.ones(len(X_unlabeled))])

lp = LabelPropagation(max_iter=1000)
lp.fit(X_combined, y_combined)
acc_lp = lp.score(X_test, y_test)
print(f"   Label Propagation: {acc_lp:.4f}")

# Semi-supervised: Label Spreading
print("\n2. Label Spreading...")
ls = LabelSpreading(max_iter=1000, alpha=0.2)
ls.fit(X_combined, y_combined)
acc_ls = ls.score(X_test, y_test)
print(f"   Label Spreading: {acc_ls:.4f}")

# Semi-supervised: Self-Training (Pseudo-labeling)
print("\n3. Self-Training (Pseudo-labeling)...")

clf_self = LogisticRegression(max_iter=1000, random_state=42)
X_train_self = X_labeled.copy()
y_train_self = y_labeled.copy()

for iteration in range(5):
    # Train on current labeled data
    clf_self.fit(X_train_self, y_train_self)
    
    # Predict on unlabeled data with confidence
    probs = clf_self.predict_proba(X_unlabeled)
    max_probs = np.max(probs, axis=1)
    predictions = clf_self.predict(X_unlabeled)
    
    # Select high-confidence predictions (>0.95)
    confident_mask = max_probs > 0.95
    
    if np.sum(confident_mask) == 0:
        break
    
    # Add pseudo-labels to training set
    X_train_self = np.vstack([X_train_self, X_unlabeled[confident_mask]])
    y_train_self = np.hstack([y_train_self, predictions[confident_mask]])
    
    # Remove from unlabeled
    X_unlabeled = X_unlabeled[~confident_mask]
    
    print(f"   Iteration {iteration+1}: Added {np.sum(confident_mask)} pseudo-labels")

acc_self = clf_self.score(X_test, y_test)
print(f"   Self-Training: {acc_self:.4f}")

# Summary
print("\n" + "="*60)
print("SEMI-SUPERVISED LEARNING RESULTS")
print("="*60)
results = pd.DataFrame({
    'Method': ['Supervised Only', 'Label Propagation', 'Label Spreading', 'Self-Training'],
    'Accuracy': [acc_supervised, acc_lp, acc_ls, acc_self],
    'Improvement': [0, acc_lp - acc_supervised, acc_ls - acc_supervised, acc_self - acc_supervised]
})
print(results.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.bar(results['Method'], results['Accuracy'], color=['gray', 'skyblue', 'lightgreen', 'salmon'])
plt.axhline(y=acc_supervised, color='red', linestyle='--', label='Supervised Baseline')
plt.ylabel('Accuracy', fontweight='bold')
plt.title('Semi-Supervised Learning Comparison\n(10% labeled data)', 
          fontweight='bold', fontsize=14)
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Semi-supervised learning dramatically improves performance with limited labels!")